# ProxyTrace 03. 주주제안 나누기 (안건 추출 + 화자 분리)

02 노트북의 결과를 읽어서, 주주제안마다 **주주의 말**과 **회사의 말**을 나눠요.

02에서 확인한 Meta 위임장 구조:
```
Proposal ○○: Shareholder Proposal Regarding ...   ← 제안 시작
The proponent(s) of this resolution is/are ...    ← 제안자
  (주주의 요구 + 이유)
META BOARD RESPONSE                               ← 회사 반론 시작
  •요약 bullet                                     ← 회사가 직접 뽑은 핵심 주장
  (자세한 설명)
The board of directors recommends a vote AGAINST  ← 회사 반론 끝
```

결과: `data/parsed/{TICKER}/{제출일}_proposals.json`

## 1. 설정 + 02 결과 불러오기

In [ ]:
import json
import re
from pathlib import Path

# ▶ 01·02 노트북과 같은 티커
TICKER = "META"

CLEAN_DIR = Path("data/clean") / TICKER.lower()
PARSED_DIR = Path("data/parsed") / TICKER.lower()
PARSED_DIR.mkdir(parents=True, exist_ok=True)

manifest = json.loads((Path("data/raw") / TICKER.lower() / "manifest.json").read_text(encoding="utf-8"))
FILING = manifest["proxies"][0]["filingDate"]          # 최신 위임장
doc = json.loads((CLEAN_DIR / f"{FILING}_pages.json").read_text(encoding="utf-8"))
print(manifest["company"], FILING, f"페이지 {len(doc['pages'])}개")

## 2. 자르는 규칙
02 노트북에서 직접 확인한 문구로 만든 규칙이에요. 다른 회사를 할 때는 이 셀에 후보 문구만 추가하면 돼요.

In [ ]:
# ── 자르는 규칙 (02 노트북에서 직접 확인한 문구) ──────────────────
# 회사마다 문구가 다르면 여기에 후보를 추가하면 돼요.

# 제안 제목: "Proposal Four: Shareholder Proposal Regarding ..."
TITLE_RE = re.compile(r"^Proposal\s+([A-Za-z\-]+)\s*[:.\-–—]\s*(.*Shareholder Proposal.*)$", re.I)

# 제안자: "The proponent of this resolution is ..." / "The proponents of this resolution are ..."
PROPONENT_RE = re.compile(r"^The proponents? of this (?:resolution|proposal) (?:is|are)\s+(.+?)\.?$", re.I)

# 회사 반론 시작: Meta는 "META BOARD RESPONSE". 다른 회사 표현도 후보로 넣어 둠
BOARD_START_RE = re.compile(
    r"^(?:META\s+)?BOARD(?:\s+OF\s+DIRECTORS)?\s+RESPONSE$"
    r"|^.{0,40}statement in opposition"
    r"|^(?:our\s+)?board(?:\s+of\s+directors)?'?s?\s+(?:response|statement|recommendation)$",
    re.I)

# 회사 반론 끝: "The board of directors recommends a vote AGAINST the shareholder proposal."
END_RE = re.compile(r"recommends a vote (AGAINST|FOR)", re.I)

WORD_NUM = {w: i for i, w in enumerate(
    "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen "
    "fifteen sixteen seventeen eighteen nineteen twenty".split())}

## 3. 함수 정의

In [ ]:
def normalize(line: str) -> str:
    """둥근 따옴표(’ “ ”)를 곧은 따옴표로 바꿔서 규칙이 안 빗나가게"""
    return line.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"').strip()


def flatten(doc: dict) -> list[dict]:
    """페이지를 줄 단위로 펼치기. 줄마다 쪽 번호를 붙여 둬서 나중에 근거 위치를 표시할 수 있어요."""
    out = []
    for p in doc["pages"]:
        for ln in p["text"].split("\n"):
            if ln.strip():
                out.append({"page_no": p["page_no"], "page_idx": p["page_idx"], "text": normalize(ln)})
    return out


def find_titles(lines: list[dict]) -> list[int]:
    """제안 제목 줄의 위치 찾기.
    목차·요약에도 같은 제목이 있을 수 있어서, 바로 뒤 3줄 안에 '제안자' 문장이 있는 것만 본문 제목으로 인정해요."""
    idx = []
    for i, ln in enumerate(lines):
        if not TITLE_RE.match(ln["text"]):
            continue
        for l in lines[i + 1:i + 4]:
            if TITLE_RE.match(l["text"]):      # 다른 제목이 먼저 나오면 목차로 판단
                break
            if PROPONENT_RE.match(l["text"]):
                idx.append(i)
                break
    return idx


def bullets_after(lines: list[dict], start: int) -> list[str]:
    """회사 반론 제목 바로 뒤의 • 요약 bullet 모으기 (= 회사가 직접 뽑은 핵심 주장)"""
    out = []
    for ln in lines[start + 1:]:
        if ln["text"].startswith("•"):
            out.append(ln["text"].lstrip("• ").strip())
        elif out:
            break
    return out


def join(block: list[dict]) -> str:
    return "\n".join(l["text"] for l in block)


def page_range(block: list[dict]) -> list:
    nos = [l["page_no"] for l in block if l["page_no"] is not None]
    return [min(nos), max(nos)] if nos else [None, None]


def split_proposals(lines: list[dict]) -> list[dict]:
    titles = find_titles(lines)
    proposals = []
    for k, t in enumerate(titles):
        nxt = titles[k + 1] if k + 1 < len(titles) else len(lines)
        seg = lines[t:nxt]
        warn = []

        m = TITLE_RE.match(seg[0]["text"])
        num_word, title = m.group(1), m.group(2).strip()
        number = WORD_NUM.get(num_word.lower(), int(num_word) if num_word.isdigit() else None)
        topic = re.sub(r"^.*Shareholder Proposal\s+(?:Regarding|on|Requesting)?\s*", "", title, flags=re.I)

        # 제안자
        proponent, prop_i = None, None
        for i, l in enumerate(seg[1:4], start=1):
            pm = PROPONENT_RE.match(l["text"])
            if pm:
                proponent, prop_i = pm.group(1).strip(), i
                break

        # 회사 반론 시작 / 끝
        b_start = next((i for i, l in enumerate(seg) if BOARD_START_RE.match(l["text"])), None)
        b_end = next((i for i, l in enumerate(seg) if END_RE.search(l["text"]) and (b_start is None or i > b_start)), None)
        if b_start is None:
            warn.append("회사 반론 시작을 못 찾음")
        if b_end is None:
            warn.append("권고 문장(끝)을 못 찾음")
            b_end = len(seg) - 1

        prop_block = seg[(prop_i or 0) + 1:(b_start if b_start is not None else b_end)]
        board_block = seg[b_start + 1:b_end] if b_start is not None else []
        recommendation = END_RE.search(seg[b_end]["text"])

        proposals.append({
            "number": number,
            "title": title,
            "topic": topic,
            "proponent": proponent,
            "pages": page_range(seg[:b_end + 1]),
            "proponent_text": join(prop_block),
            "proponent_pages": page_range(prop_block),
            "board_summary": bullets_after(seg, b_start) if b_start is not None else [],
            "board_text": join(board_block),
            "board_pages": page_range(board_block),
            "recommendation": recommendation.group(1).upper() if recommendation else None,
            "warnings": warn,
        })
    return proposals

## 4. 나누기 실행

In [ ]:
lines = flatten(doc)
proposals = split_proposals(lines)
print(f"전체 {len(lines)}줄 → 주주제안 {len(proposals)}개")

## 5. 결과 확인 ⚠ 중요
- 제안이 **10개**(3~12번)인지
- 제안자 이름이 다 나왔는지
- 경고(warnings)가 없는지
- 주주 글자 수와 회사 글자 수가 둘 다 0보다 큰지

In [ ]:
print(f"{'번호':>3} | {'쪽':^9} | {'주주':>6} | {'회사':>6} | {'요약':>2} | 권고    | 제목 / 제안자")
print("-" * 110)
for p in proposals:
    pg = f"{p['pages'][0]}-{p['pages'][1]}"
    print(f"{p['number']:>3} | {pg:^9} | {len(p['proponent_text']):>6} | {len(p['board_text']):>6} | "
          f"{len(p['board_summary']):>2}개 | {p['recommendation'] or '-':7} | {p['topic'][:45]}")
    print(f"{'':>3}   {'':9}   {'':6}   {'':6}   {'':4}   {'':7}   └ {p['proponent']}")
    if p["warnings"]:
        print(f"      ⚠ {p['warnings']}")

In [ ]:
# 제안 하나를 자세히 보기 (번호만 바꾸면 돼요)
def show(number: int, n_chars: int = 800):
    p = next(x for x in proposals if x["number"] == number)
    print(f"[{p['number']}] {p['title']}")
    print(f"제안자: {p['proponent']}")
    print(f"\n── 주주의 말 (p.{p['proponent_pages'][0]}-{p['proponent_pages'][1]}) ──")
    print(p["proponent_text"][:n_chars], "...")
    print(f"\n── 회사 요약 bullet ──")
    for b in p["board_summary"]:
        print(" •", b)
    print(f"\n── 회사의 말 (p.{p['board_pages'][0]}-{p['board_pages'][1]}) ──")
    print(p["board_text"][:n_chars], "...")

show(4)

## 6. 저장

In [ ]:
out = {
    "ticker": TICKER,
    "filingDate": FILING,
    "source_url": manifest["proxies"][0]["url"],
    "proposals": proposals,
}
path = PARSED_DIR / f"{FILING}_proposals.json"
path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"저장: {path}")

## 다음 단계
5번 셀 표를 확인하고 보내 주세요. 10개가 다 잘 나뉘면 `04_extract_claims.ipynb`에서 **주장 추출(LLM)**을 시작해요.

경고가 뜬 제안이 있으면 `show(번호)`로 열어 보고 그 부분을 보내 주세요.